In [1]:
!pip install -q "monai[nibabel,tqdm]" pydicom highdicom SimpleITK

import os
from monai.apps import DecathlonDataset

os.makedirs("/kaggle/working/data", exist_ok=True)

dataset = DecathlonDataset(
    root_dir="/kaggle/working/data",
    task="Task09_Spleen",
    section="training",
    download=True,
    transform=None,
)

print("Dataset ready.")
print("Training cases:", len(dataset))

Task09_Spleen.tar: 1.50GB [01:03, 25.2MB/s]                               


2026-10-27 19:39:04,016 - INFO - Verified 'Task09_Spleen.tar', md5: 410d4a301da4e5b2f6f86ec3ddba524e.
2026-10-27 19:39:05,115 - INFO - Downloaded: /kaggle/working/data/Task09_Spleen.tar
2026-10-27 19:39:05,117 - INFO - Writing into directory: /kaggle/working/data.


Loading dataset: 100%|██████████| 33/33 [00:00<00:00, 101252.40it/s]

Dataset ready.
Training cases: 33


In [3]:
from pathlib import Path
import SimpleITK as sitk

image_paths = sorted(
    path
    for path in Path(
        "/kaggle/working/data/Task09_Spleen/imagesTr"
    ).glob("*.nii.gz")
    if not path.name.startswith("._")
)

scan_path = image_paths[0]
scan = sitk.ReadImage(str(scan_path))

print("File:", scan_path.name)
print("Size (x, y, z):", scan.GetSize())
print("Spacing in mm:", scan.GetSpacing())
print("Origin:", scan.GetOrigin())
print("Direction:", scan.GetDirection())

File: spleen_10.nii.gz
Size (x, y, z): (512, 512, 55)
Spacing in mm: (0.9765620231628418, 0.9765620231628418, 5.0)
Origin: (499.023193359375, 499.023193359375, 0.0)
Direction: (-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)


In [4]:
import numpy as np

pixels = sitk.GetArrayFromImage(scan)

print("Array shape (z, y, x):", pixels.shape)
print("Pixel data type:", pixels.dtype)
print("Minimum value:", pixels.min())
print("Maximum value:", pixels.max())
print("All values are whole numbers:",
      np.allclose(pixels, np.rint(pixels), rtol=0, atol=1e-6))

Array shape (z, y, x): (55, 512, 512)
Pixel data type: float32
Minimum value: -1024.0
Maximum value: 3072.0
All values are whole numbers: True


In [5]:
from datetime import datetime
from pydicom.uid import generate_uid

# 1. Change storage type while keeping the scan's geometry
dicom_scan = sitk.Cast(scan, sitk.sitkInt16)

out_dir = Path("/kaggle/working/dicom_in/case_0")
out_dir.mkdir(parents=True, exist_ok=True)

# Prevent mixing a new conversion with files from an earlier run
if list(out_dir.glob("*.dcm")):
    raise RuntimeError(
        "This folder already contains DICOM files. "
        "Stop here and tell me before rerunning."
    )

# Format numeric DICOM tags without unnecessarily long decimals
def dicom_numbers(values):
    return "\\".join(f"{value:.10g}" for value in values)

# 2. These identifiers are shared by every slice
study_uid = generate_uid()
series_uid = generate_uid()
frame_uid = generate_uid()

now = datetime.now()
date = now.strftime("%Y%m%d")
time = now.strftime("%H%M%S")

direction = dicom_scan.GetDirection()
orientation = (
    direction[0], direction[3], direction[6],
    direction[1], direction[4], direction[7],
)

shared_tags = {
    "0008|0016": "1.2.840.10008.5.1.4.1.1.2",  # CT Image Storage
    "0008|0008": "DERIVED\\SECONDARY\\AXIAL",
    "0008|0060": "CT",
    "0008|0020": date,
    "0008|0030": time,
    "0008|0050": "",                           # Unknown accession number
    "0008|0090": "",                           # Unknown referring physician
    "0008|103e": "MSD Spleen - simulated CT",
    "0008|2111": "Converted from research NIfTI for pipeline testing",
    "0010|0010": "DEMO^CASE0",                  # Synthetic patient name
    "0010|0020": "DEMO_CASE0",
    "0010|0030": "",                           # Unknown birth date
    "0010|0040": "",                           # Unknown sex
    "0020|000d": study_uid,
    "0020|000e": series_uid,
    "0020|0052": frame_uid,
    "0020|0010": "DEMO",
    "0020|0011": "1",
    "0020|0037": dicom_numbers(orientation),
    "0020|1040": "",
    "0018|0050": str(dicom_scan.GetSpacing()[2]),
    "0028|1052": "0",                          # Rescale intercept
    "0028|1053": "1",                          # Rescale slope
    "0028|1054": "HU",
}

writer = sitk.ImageFileWriter()
writer.KeepOriginalImageUIDOn()

# 3. Save one DICOM file per slice
for z in range(dicom_scan.GetDepth()):
    slice_image = dicom_scan[:, :, z]

    for tag, value in shared_tags.items():
        slice_image.SetMetaData(tag, value)

    slice_image.SetMetaData("0008|0018", generate_uid())
    slice_image.SetMetaData("0020|0013", str(z + 1))

    position = dicom_scan.TransformIndexToPhysicalPoint((0, 0, z))
    slice_image.SetMetaData("0020|0032", dicom_numbers(position))

    writer.SetFileName(str(out_dir / f"{z:04d}.dcm"))
    writer.Execute(slice_image)

print("DICOM files written:", len(list(out_dir.glob("*.dcm"))))
print("Saved to:", out_dir)

DICOM files written: 55
Saved to: /kaggle/working/dicom_in/case_0


In [6]:
# Find the series and let the reader order its slices
reader = sitk.ImageSeriesReader()

files = reader.GetGDCMSeriesFileNames(str(out_dir))
reader.SetFileNames(files)
restored = reader.Execute()

# Compare the reconstructed scan with the original
checks = {
    "55 slices": len(files) == scan.GetDepth(),
    "Same size": restored.GetSize() == scan.GetSize(),
    "Same spacing": np.allclose(
        restored.GetSpacing(), scan.GetSpacing(), rtol=0, atol=1e-5
    ),
    "Same origin": np.allclose(
        restored.GetOrigin(), scan.GetOrigin(), rtol=0, atol=1e-5
    ),
    "Same direction": np.allclose(
        restored.GetDirection(), scan.GetDirection(), rtol=0, atol=1e-5
    ),
    "Exactly the same pixel values": np.array_equal(
        sitk.GetArrayFromImage(restored),
        sitk.GetArrayFromImage(scan),
    ),
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

55 slices: PASS
Same size: PASS
Same spacing: PASS
Same origin: PASS
Same direction: PASS
Exactly the same pixel values: PASS


## Conversion result

Converted spleen_10.nii.gz into 55 simulated CT DICOM slices.

Verified by reading the series back:
- Image size, spacing, origin, and direction match the source.
- All pixel values match exactly.

These files were generated from research NIfTI data.
Original scanner metadata is not available.
DICOM standards conformance has not yet been independently validated.